In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from collections import Counter
import re
import torch
import torch.nn as nn
import math
import torch.nn.functional as F

/Users/nawinkmr/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nawinkmr/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_credits = pd.read_csv('../archive/credits.csv')
data_movies = pd.read_csv('../archive/movies_metadata.csv')
data_keywords = pd.read_csv('../archive/keywords.csv')
data_links = pd.read_csv('../archive/links.csv')
data_links_small = pd.read_csv('../archive/links_small.csv')
data_ratings = pd.read_csv('../archive/ratings_small.csv')

/var/folders/f8/mk8n2xqj0qx1p_8wvgzyzbsc0000gp/T/ipykernel_2306/2249517742.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_movies = pd.read_csv('../archive/movies_metadata.csv')


In [3]:
data_credits

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862
...,...,...,...
45471,"[{'cast_id': 0, 'character': '', 'credit_id': ...","[{'credit_id': '5894a97d925141426c00818c', 'de...",439050
45472,"[{'cast_id': 1002, 'character': 'Sister Angela...","[{'credit_id': '52fe4af1c3a36847f81e9b15', 'de...",111109
45473,"[{'cast_id': 6, 'character': 'Emily Shaw', 'cr...","[{'credit_id': '52fe4776c3a368484e0c8387', 'de...",67758
45474,"[{'cast_id': 2, 'character': '', 'credit_id': ...","[{'credit_id': '533bccebc3a36844cf0011a7', 'de...",227506


In [4]:
data_movies.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [5]:
data_movies.drop(['belongs_to_collection' , 'budget' , 'homepage' , 'imdb_id' , 'original_language' , 'original_title' , 'revenue' , 'runtime' , 'spoken_languages' , 'status' , 'video'] , inplace = True , axis = 1)

In [6]:
data_keywords.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46419 entries, 0 to 46418
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        46419 non-null  int64 
 1   keywords  46419 non-null  object
dtypes: int64(1), object(1)
memory usage: 725.4+ KB


In [7]:
data_links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [8]:
data_ratings

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205
...,...,...,...,...
99999,671,6268,2.5,1065579370
100000,671,6269,4.0,1065149201
100001,671,6365,4.0,1070940363
100002,671,6385,2.5,1070979663


In [9]:
def clean_cast(cast):

    keys= ast.literal_eval(cast)
    key = []
    for item in keys:
        key.append(item['name'])
    return key

In [10]:
data_credits['cast'] = data_credits['cast'].apply(clean_cast)
data_credits.head()

,cast,crew,id
0,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[Whitney Houston, Angela Bassett, Loretta Devi...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[Steve Martin, Diane Keaton, Martin Short, Kim...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [11]:
def clean_crew(crew):

    keys= ast.literal_eval(crew)
    key = []
    for item in keys:
        key.append(item['name'])
    return key

In [12]:
data_credits['crew'] = data_credits['crew'].apply(clean_crew)
data_credits.head()

,cast,crew,id
0,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...","[John Lasseter, Joss Whedon, Andrew Stanton, J...",862
1,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...","[Larry J. Franco, Jonathan Hensleigh, James Ho...",8844
2,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...","[Howard Deutch, Mark Steven Johnson, Mark Stev...",15602
3,"[Whitney Houston, Angela Bassett, Loretta Devi...","[Forest Whitaker, Ronald Bass, Ronald Bass, Ez...",31357
4,"[Steve Martin, Diane Keaton, Martin Short, Kim...","[Alan Silvestri, Elliot Davis, Nancy Meyers, N...",11862


In [13]:
data_keywords['keywords'][0]

"[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'friendship'}, {'id': 9713, 'name': 'friends'}, {'id': 9823, 'name': 'rivalry'}, {'id': 165503, 'name': 'boy next door'}, {'id': 170722, 'name': 'new toy'}, {'id': 187065, 'name': 'toy comes to life'}]"

In [14]:
def clean_keywords(keywords):

    keys= ast.literal_eval(keywords)
    key = []
    for item in keys:
        key.append(item['name'])
    return key

In [15]:
data_keywords['keywords'] = data_keywords['keywords'].apply(clean_keywords)
data_keywords.head()

,id,keywords
0,862,"[jealousy, toy, boy, friendship, friends, riva..."
1,8844,"[board game, disappearance, based on children'..."
2,15602,"[fishing, best friend, duringcreditsstinger, o..."
3,31357,"[based on novel, interracial relationship, sin..."
4,11862,"[baby, midlife crisis, confidence, aging, daug..."


In [16]:
def clean_genre(genre):

    keys= ast.literal_eval(genre)
    key = []
    for item in keys:
        key.append(item['name'])
    return key

In [17]:
data_movies['genres'] = data_movies['genres'].apply(clean_genre)
data_movies.head()

,adult,genres,id,overview,popularity,poster_path,production_companies,production_countries,release_date,tagline,title,vote_average,vote_count
0,False,"[Animation, Comedy, Family]",862,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,NaN,Toy Story,7.7,5415.0
1,False,"[Adventure, Fantasy, Family]",8844,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0
2,False,"[Romance, Comedy]",15602,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0
3,False,"[Comedy, Drama, Romance]",31357,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0
4,False,[Comedy],11862,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0


In [18]:
data_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   adult                 45466 non-null  object 
 1   genres                45466 non-null  object 
 2   id                    45466 non-null  object 
 3   overview              44512 non-null  object 
 4   popularity            45461 non-null  object 
 5   poster_path           45080 non-null  object 
 6   production_companies  45463 non-null  object 
 7   production_countries  45463 non-null  object 
 8   release_date          45379 non-null  object 
 9   tagline               20412 non-null  object 
 10  title                 45460 non-null  object 
 11  vote_average          45460 non-null  float64
 12  vote_count            45460 non-null  float64
dtypes: float64(2), object(11)
memory usage: 4.5+ MB


In [19]:
def clean_production_companies(value):
    if pd.isna(value):
        return []

    if type(value) == bool:
        return []

    try:
        keys = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

    key = []

    if type(keys) == list:
        for item in keys:
            if type(item) == dict and 'name' in item:
                key.append(item['name'])

    return key

In [20]:
data_movies['production_companies'] = data_movies['production_companies'].apply(clean_production_companies)
data_movies.head()

,adult,genres,id,overview,popularity,poster_path,production_companies,production_countries,release_date,tagline,title,vote_average,vote_count
0,False,"[Animation, Comedy, Family]",862,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,[Pixar Animation Studios],"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,NaN,Toy Story,7.7,5415.0
1,False,"[Adventure, Fantasy, Family]",8844,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[TriStar Pictures, Teitler Film, Interscope Co...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0
2,False,"[Romance, Comedy]",15602,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[Warner Bros., Lancaster Gate]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0
3,False,"[Comedy, Drama, Romance]",31357,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[Twentieth Century Fox Film Corporation],"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0
4,False,[Comedy],11862,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[Sandollar Productions, Touchstone Pictures]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0


In [21]:
def clean_production_countries(value):
    if pd.isna(value):
        return []

    if type(value) == bool:
        return []

    try:
        keys = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

    key = []

    if type(keys) == list:
        for item in keys:
            if type(item) == dict and 'name' in item:
                key.append(item['name'])

    return key

In [22]:
data_movies['production_countries'] = data_movies['production_countries'].apply(clean_production_countries)
data_movies.head()

,adult,genres,id,overview,popularity,poster_path,production_companies,production_countries,release_date,tagline,title,vote_average,vote_count
0,False,"[Animation, Comedy, Family]",862,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,[Pixar Animation Studios],[United States of America],1995-10-30,NaN,Toy Story,7.7,5415.0
1,False,"[Adventure, Fantasy, Family]",8844,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[TriStar Pictures, Teitler Film, Interscope Co...",[United States of America],1995-12-15,Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0
2,False,"[Romance, Comedy]",15602,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[Warner Bros., Lancaster Gate]",[United States of America],1995-12-22,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0
3,False,"[Comedy, Drama, Romance]",31357,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[Twentieth Century Fox Film Corporation],[United States of America],1995-12-22,Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0
4,False,[Comedy],11862,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[Sandollar Productions, Touchstone Pictures]",[United States of America],1995-02-10,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0


In [23]:
data_movies['tags'] = (
    data_movies['overview'].fillna('') + ' ' +
    data_movies['genres'].apply(lambda x: ' '.join(x)) + ' ' +
    data_movies['production_companies'].apply(lambda x: ' '.join(x)) + ' ' +
    data_movies['production_countries'].apply(lambda x: ' '.join(x)) + ' ' +
    data_movies['tagline'].fillna('').apply(lambda x: ' '.join(x)) + ' ' +
    data_movies['title'].fillna('').apply(lambda x: ' '.join(x)) + ' ' +
    data_credits['cast'].apply(lambda x: ' '.join(x)) + ' ' +
    data_credits['crew'].apply(lambda x: ' '.join(x)) + ' '
)

In [24]:
data_movies['tags'][0]

"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Animation Comedy Family Pixar Animation Studios United States of America  T o y   S t o r y Tom Hanks Tim Allen Don Rickles Jim Varney Wallace Shawn John Ratzenberger Annie Potts John Morris Erik von Detten Laurie Metcalf R. Lee Ermey Sarah Freeman Penn Jillette John Lasseter Joss Whedon Andrew Stanton Joel Cohen Alec Sokolow Bonnie Arnold Ed Catmull Ralph Guggenheim Steve Jobs Lee Unkrich Ralph Eggleston Robert Gordon Mary Helen Leasman Kim Blanchette Marilyn McCoppen Randy Newman Dale E. Grahn Robin Cooper John Lasseter Pete Docter Joe Ranft Patsy Bouge Norm DeCarlo Ash Brannon Randy Newman Roman Figun Don Davis James Flamberg Mary Beth Smith Rick Mackay Susan Bradley William Reeves R

In [25]:
data_movies['tags'] = (
    data_movies['tags']
    .fillna('')
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [26]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english')

tfidf_matrix = tfidf.fit_transform(data_movies['tags'])
print("TFIDF matrix shape:", tfidf_matrix.shape)

TFIDF matrix shape: (45466, 10000)


In [27]:
def similarity_score(movie_title , top_k = 5):
    matches = data_movies.index[data_movies['title'].str.casefold() == movie_title.casefold()].to_list()
    
    if not matches:
        print("No matches found for the given movie title.")
    
    movie_index = matches[0]
    scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).ravel()
    
    scores[movie_index] = -1
    similar_movies_indices = np.argsort(scores)[-top_k:][::-1]
    
    results = data_movies.iloc[similar_movies_indices][['title', 'release_date']].copy()
    results['similarity_score'] = scores[similar_movies_indices]
    return results

In [28]:
similarity_score("the godfather", top_k=5)

,title,release_date,similarity_score
1178,The Godfather: Part II,1974-12-20,0.569485
1914,The Godfather: Part III,1990-12-24,0.471477
1165,Apocalypse Now,1979-08-15,0.370845
8970,Hearts of Darkness: A Filmmaker's Apocalypse,1991-09-06,0.335659
5971,One from the Heart,1982-02-12,0.304602


In [29]:
texts = data_movies['tags']

def tokenize(text):
    return re.findall(r"\b\w+\b|[^\w\s]", text.lower())

sample_text = texts.iloc[0]
sample_tokens = tokenize(sample_text)

print("Original text:" , sample_text)
print("Tokenized text:" , sample_tokens)

Original text: led by woody, andy's toys live happily in his room until andy's birthday brings buzz lightyear onto the scene. afraid of losing his place in andy's heart, woody plots against buzz. but when circumstances separate buzz and woody from their owner, the duo eventually learns to put aside their differences. animation comedy family pixar animation studios united states of america t o y s t o r y tom hanks tim allen don rickles jim varney wallace shawn john ratzenberger annie potts john morris erik von detten laurie metcalf r. lee ermey sarah freeman penn jillette john lasseter joss whedon andrew stanton joel cohen alec sokolow bonnie arnold ed catmull ralph guggenheim steve jobs lee unkrich ralph eggleston robert gordon mary helen leasman kim blanchette marilyn mccoppen randy newman dale e. grahn robin cooper john lasseter pete docter joe ranft patsy bouge norm decarlo ash brannon randy newman roman figun don davis james flamberg mary beth smith rick mackay susan bradley willi

In [30]:
split_index = int(len(texts) * 0.9)
train_texts = texts.iloc[:split_index]
val_texts = texts.iloc[split_index:]

In [31]:
token_counts = Counter(
 token for text in train_texts for token in tokenize(text)   
)

vocab = {
    '<PAD>': 0,
    '<UNK>': 1,
    '<MASK>': 2,
    '<BOS>': 3,
    '<EOS>': 4
}

max_vocab_size = 20000

for token , count in token_counts.most_common(max_vocab_size - len(vocab)):
    vocab[token] = len(vocab)

print("Vocabulary size:", len(vocab))

Vocabulary size: 20000


In [32]:
PAD_ID = vocab['<PAD>']
UNK_ID = vocab['<UNK>']
MASK_ID = vocab['<MASK>']
BOS_ID = vocab['<BOS>']
EOS_ID = vocab['<EOS>']

def encode(text):
    tokens = tokenize(text)
    token_ids = [BOS_ID]
    for token in tokens:
        if token not in vocab:
            token_ids.append(UNK_ID)
        else:
            token_ids.append(vocab[token])
    token_ids.append(EOS_ID)
    return token_ids

In [33]:
max_length = 512

def encode_with_padding(text):
    token_ids = encode(text)
    if len(token_ids) > max_length:
        token_ids = token_ids[:max_length]
        token_ids[-1] = EOS_ID  

    attention_mask = [1] * len(token_ids)
    token_ids += [PAD_ID] * (max_length - len(token_ids))
    attention_mask += [0] * (max_length - len(attention_mask))
    
    return token_ids, attention_mask

In [34]:
token_ids , attention_mask = encode_with_padding(data_movies['tags'].iloc[0])

print('Tokens:', token_ids)
print('Attention Mask:', attention_mask)
print('Length of Tokens:', len(token_ids))
print('Length of Attention Mask:', len(attention_mask))

Tokens: [3, 1278, 50, 1586, 8, 341, 23, 11, 7501, 416, 4312, 24, 29, 1351, 473, 341, 23, 11, 2768, 1209, 4989, 1, 3724, 9, 1756, 6, 6649, 16, 2328, 29, 507, 24, 341, 23, 11, 862, 8, 1586, 4752, 323, 4989, 6, 65, 66, 3122, 3942, 4989, 18, 1586, 61, 59, 958, 8, 9, 4116, 1044, 900, 19, 844, 8815, 59, 4922, 6, 242, 57, 87, 8948, 242, 583, 35, 40, 16, 38, 10, 12, 32, 11, 10, 12, 14, 32, 101, 3629, 173, 277, 163, 14382, 158, 13355, 706, 880, 41, 10664, 1056, 8816, 41, 570, 705, 415, 1, 1405, 9408, 14, 6, 98, 12215, 261, 920, 1937, 19577, 41, 5821, 4515, 14720, 138, 2244, 401, 630, 1433, 1, 1265, 600, 405, 1, 413, 13059, 128, 3220, 98, 1, 413, 19578, 58, 229, 154, 480, 1, 180, 1, 1658, 1, 758, 721, 841, 7, 6, 13682, 427, 565, 41, 5821, 1072, 16025, 129, 11701, 4682, 1, 5822, 1, 3159, 1, 758, 721, 1106, 1, 163, 245, 64, 1, 154, 1082, 127, 365, 4549, 305, 913, 84, 2133, 758, 721, 138, 2244, 1072, 16025, 175, 13060, 400, 58, 353, 120, 1, 515, 1, 49, 1, 526, 1818, 4956, 8019, 1, 2983, 1, 2248, 1,

In [35]:
encoded_data = [encode_with_padding(text) for text in train_texts]

input_ids = torch.tensor([item[0] for item in encoded_data])
attention_ids = torch.tensor([item[1] for item in encoded_data])

print("Input IDs shape:", input_ids.shape)
print("Attention IDs shape:", attention_ids.shape)

Input IDs shape: torch.Size([40919, 512])
Attention IDs shape: torch.Size([40919, 512])


In [36]:
vocab_size = len(vocab)
embedding_dim = 512
max_length = 512

token_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
positional_embeddings = nn.Embedding(max_length, embedding_dim)

In [37]:
x = input_ids[:32]

token_vectors = token_embeddings(x)
positions = torch.arange(max_length)

positional_vectors = positional_embeddings(positions)

embeddings = token_vectors + positional_vectors

print("Input Shape:", x.shape)
print("Embeddings shape:", embeddings.shape)
print("Token Vectors shape:", token_vectors.shape)
print("Positional Vectors shape:", positional_vectors.shape)

Input Shape: torch.Size([32, 512])
Embeddings shape: torch.Size([32, 512, 512])
Token Vectors shape: torch.Size([32, 512, 512])
Positional Vectors shape: torch.Size([512, 512])


In [38]:
class multiheadattention(nn.Module):
    
    def __init__(self , embedding_dim , num_heads):
        super().__init__()
        self.query = nn.Linear(embedding_dim , embedding_dim)
        self.key = nn.Linear(embedding_dim , embedding_dim)
        self.value = nn.Linear(embedding_dim , embedding_dim)
        
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        
        self.fc_out = nn.Linear(embedding_dim , embedding_dim)
        
    def forward(self , x , attention_mask):
        batch_size , seq_length , embedding_dim = x.shape
        
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        
        Q = Q.view(batch_size , seq_length , self.num_heads , self.head_dim).transpose(1,2)
        K = K.view(batch_size , seq_length , self.num_heads , self.head_dim).transpose(1,2)
        V = V.view(batch_size , seq_length , self.num_heads , self.head_dim).transpose(1,2)
        
        scores = (Q @ K.transpose(-2, -1))/math.sqrt(self.head_dim)
        
        scores = scores.masked_fill(
            attention_mask[:, None,None,:] == 0,
            float('-inf')
        )
        
        attention_weights = F.softmax(scores , dim = -1)
        
        output = attention_weights @ V
        output = output.transpose(1,2).contiguous()
        output = output.view(batch_size , seq_length , embedding_dim)
        
        return self.fc_out(output)

In [39]:
attention = multiheadattention(embedding_dim=512,
                               num_heads = 8)

x = input_ids[:32]
mask = attention_ids[:32]

token_vectors = token_embeddings(x)

positions = torch.arange(max_length)
position_vectors = positional_embeddings(positions)

embeddings = token_vectors + position_vectors

attention_output = attention(embeddings, mask)

print("Output shape:", attention_output.shape)

Output shape: torch.Size([32, 512, 512])


In [40]:
class AttentionBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()

        self.attention = multiheadattention(embedding_dim, num_heads)
        self.norm = nn.LayerNorm(embedding_dim)

    def forward(self, x, attention_mask):
        attention_output = self.attention(x, attention_mask)

        # Residual connection + layer normalization
        output = self.norm(x + attention_output)

        return output

In [41]:
attention_block = AttentionBlock(
    embedding_dim=512,
    num_heads=8
)

block_output = attention_block(embeddings, mask)

print("Input shape:", embeddings.shape)
print("Output shape:", block_output.shape)

Input shape: torch.Size([32, 512, 512])
Output shape: torch.Size([32, 512, 512])


In [42]:
class FFNN(nn.Module):
    def __init__(self , embedding_dim):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(embedding_dim , embedding_dim * 4),
            nn.ReLU(),
            nn.Linear(embedding_dim * 4 , embedding_dim)
        )
        self.norm = nn.LayerNorm(embedding_dim)
        
    def forward(self , x):
        output = self.network(x)
        output = self.norm(x + output)
        
        return output

In [43]:
feed_forward = FFNN(embedding_dim=512)

ffn_output = feed_forward(block_output)

print("Input shape:", block_output.shape)
print("Output shape:", ffn_output.shape)

Input shape: torch.Size([32, 512, 512])
Output shape: torch.Size([32, 512, 512])


In [44]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, dropout=0.1):
        super().__init__()

        self.attention = multiheadattention(embedding_dim, num_heads)
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.dropout1 = nn.Dropout(dropout)

        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 4),
            nn.ReLU(),
            nn.Linear(embedding_dim * 4, embedding_dim)
        )

        self.norm2 = nn.LayerNorm(embedding_dim)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, attention_mask):
        attention_output = self.attention(x, attention_mask)
        x = self.norm1(x + self.dropout1(attention_output))

        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_output))

        return x

In [45]:
encoder_block = TransformerEncoderBlock(
    embedding_dim=512,
    num_heads=8
)

encoder_output = encoder_block(embeddings, mask)

print("Input shape:", embeddings.shape)
print("Output shape:", encoder_output.shape)

Input shape: torch.Size([32, 512, 512])
Output shape: torch.Size([32, 512, 512])


In [46]:
class TransformerEncoder(nn.Module):
    def __init__(self, embedding_dim, num_heads, num_layers):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerEncoderBlock(embedding_dim, num_heads)
            for _ in range(num_layers)
        ])

    def forward(self, x, attention_mask):
        for layer in self.layers:
            x = layer(x, attention_mask)

        return x

In [47]:
encoder = TransformerEncoder(
    embedding_dim=512,
    num_heads=8,
    num_layers=4
)

encoder_output = encoder(embeddings, mask)

print("Input shape:", embeddings.shape)
print("Output shape:", encoder_output.shape)
print("Number of encoder blocks:", len(encoder.layers))

Input shape: torch.Size([32, 512, 512])
Output shape: torch.Size([32, 512, 512])
Number of encoder blocks: 4


In [48]:
class CineMindTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=512,
        num_heads=8,
        num_layers=4,
        max_length=512
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(max_length, embedding_dim)

        self.encoder = TransformerEncoder(
            embedding_dim,
            num_heads,
            num_layers
        )

    def forward(self, input_ids, attention_mask):
        seq_length = input_ids.shape[1]

        positions = torch.arange(
            seq_length,
            device=input_ids.device
        )

        x = self.token_embedding(input_ids)
        x = x + self.position_embedding(positions)

        return self.encoder(x, attention_mask)

In [51]:
model = CineMindTransformer(
    vocab_size=len(vocab),
    embedding_dim=512,
    num_heads=8,
    num_layers=4,
    max_length=512
)

model.eval()

with torch.no_grad():
    output = model(input_ids[:2], attention_ids[:2])

print("Output shape:", output.shape)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

Output shape: torch.Size([2, 512, 512])
Total parameters: 23111680


In [54]:
new_df = pd.DataFrame()
new_df['title'] = data_movies['title']
new_df['tags'] = data_movies['tags']
new_df['id'] = data_movies['id']
new_df

,title,tags,id
0,Toy Story,"led by woody, andy's toys live happily in his ...",862
1,Jumanji,when siblings judy and peter discover an encha...,8844
2,Grumpier Old Men,a family wedding reignites the ancient feud be...,15602
3,Waiting to Exhale,"cheated on, mistreated and stepped on, the wom...",31357
4,Father of the Bride Part II,just when george banks has recovered from his ...,11862
...,...,...,...
45461,Subdue,rising and falling between a man and woman. dr...,439050
45462,Century of Birthing,an artist struggles to finish his work while a...,111109
45463,Betrayal,"when one of her hits goes wrong, a professiona...",67758
45464,Satan Triumphant,"in a small town live two brothers, one a minis...",227506
